# ECON 1307 --- Clase 10
## Agrupación y Agregación de Datos con `groupby()`

**Profesor:** Santiago Neira

**Objetivo:** Aprender a usar el método `.groupby()` de Pandas para:
1. Dividir datos en grupos según una o más variables categóricas
2. Aplicar funciones de agregación (suma, media, conteo, etc.) a cada grupo
3. Combinar múltiples agregaciones con `.agg()`
4. Analizar datos reales de **accidentes de tránsito en Bogotá (2016)**

---

In [28]:
import pandas as pd
import numpy as np

## 1. Introducción a `groupby()`

Uno de los métodos más útiles para los analistas de datos es `.groupby()`. Este método implementa el paradigma **split-apply-combine**:

1. **Split (dividir):** Separa el DataFrame en grupos según los valores de una columna
2. **Apply (aplicar):** Ejecuta una función de agregación sobre cada grupo
3. **Combine (combinar):** Reúne los resultados en un nuevo DataFrame

Veamos el siguiente ejemplo para entender este concepto mejor:

In [29]:
df = pd.read_excel("C:/Users/David/Downloads/Clase 10/data/ejemplo_groupby.xlsx")
df

,animal,age,weight,length
0,hamster,1,7,8
1,alligator,9,13,6
2,hamster,4,8,9
3,cat,13,12,1
4,snake,14,11,8
5,cat,10,8,9
6,hamster,2,10,5
7,cat,4,14,6
8,cat,14,9,6
9,snake,7,11,6


In [30]:
# ¿Qué animales hay en la base?
print("Animales únicos:", df.animal.unique())
print("Cantidad:", df.animal.nunique())

Animales únicos: <StringArray>
['hamster', 'alligator', 'cat', 'snake']
Length: 4, dtype: str
Cantidad: 4


Note que tenemos un `DataFrame` con cuatro tipos de animales: 
- alligators (cocodrilos) 
- cats (gatos)
- snakes (serpientes)
- hamsters (hamsters)

Cada fila representa un chequeo en el veterinario donde se registra edad, peso y largo del animal. Como investigador, usted quiere estudiar estadísticas descriptivas **por especie**. Por ejemplo: **¿Cuál es el peso promedio de cada especie?**

### 1.1 Crear los grupos

In [31]:
# El primer paso es agrupar por animal
animal_groups = df.groupby("animal")
print(type(animal_groups))

<class 'pandas.api.typing.DataFrameGroupBy'>


In [32]:
# El objeto GroupBy no muestra datos directamente. 
# Podemos ver qué filas pertenecen a cada grupo:
animal_groups.groups

{'alligator': [1, 13], 'cat': [3, 5, 7, 8, 12], 'hamster': [0, 2, 6, 10, 11], 'snake': [4, 9]}

In [33]:
# También podemos ver cuántos elementos hay en cada grupo
animal_groups.size()

animal
alligator    2
cat          5
hamster      5
snake        2
dtype: int64

### 1.2 Visualización del proceso split-apply-combine

Visualmente, lo que sucedió fue lo siguiente:

1. Se agrupa los valores únicos de la columna animal.
<center>
<img src = "./img/groupby1.jpg" width = "400">
</center>

2. La segmentación de cada grupo se vería de la siguiente manera
<center>
<img src = "./img/groupby2.jpg" width = "400">
</center>

3. Se le asignan las otras variables/columnas a cada grupo
<center>
<img src = "./img/groupby3.jpg" width = "400">
</center>

4. Se aplica la función agregadora `.mean()` sobre la columna `weight` de cada grupo.
<center>
<img src = "./img/groupby4.jpg" width = "400">
</center>

### 1.3 Funciones de agregación comunes

Una vez creado el objeto GroupBy, podemos aplicar funciones de agregación:

In [34]:
# Peso promedio por especie
df.groupby("animal")["weight"].mean()

animal
alligator    13.5
cat          10.4
hamster       9.0
snake        11.0
Name: weight, dtype: float64

In [35]:
# Suma total de peso por especie
df.groupby("animal")["weight"].sum()

animal
alligator    27
cat          52
hamster      45
snake        22
Name: weight, dtype: int64

In [36]:
# Estadísticas descriptivas completas por especie
df.groupby("animal")[["weight", "length"]].describe()

weight                                                 length       \
           count  mean       std   min    25%   50%    75%   max  count mean   
animal                                                                         
alligator    2.0  13.5  0.707107  13.0  13.25  13.5  13.75  14.0    2.0  5.5   
cat          5.0  10.4  2.509980   8.0   9.00   9.0  12.00  14.0    5.0  5.2   
hamster      5.0   9.0  1.414214   7.0   8.00  10.0  10.00  10.0    5.0  6.0   
snake        2.0  11.0  0.000000  11.0  11.00  11.0  11.00  11.0    2.0  7.0   

                                                
                std  min   25%  50%   75%  max  
animal                                          
alligator  0.707107  5.0  5.25  5.5  5.75  6.0  
cat        2.949576  1.0  4.00  6.0  6.00  9.0  
hamster    2.449490  3.0  5.00  5.0  8.00  9.0  
snake      1.414214  6.0  6.50  7.0  7.50  8.0

---
## 2. Método `.agg()` --- Agregaciones personalizadas

El método `.agg()` permite aplicar **múltiples funciones** a **múltiples columnas** en una sola operación. Es mucho más flexible que usar `.mean()`, `.sum()`, etc. por separado.

La sintaxis general es:
```python
df.groupby(columnas).agg(funciones)
```

Hay **tres formas** principales de usar `.agg()`:

### 2.1 Lista de funciones (se aplican a todas las columnas seleccionadas)

In [37]:
# 2.1 Lista de funciones: min, mean, max para peso y longitud
df.groupby("animal")[['weight', 'length']].agg(["min", "mean", "max"])

weight           length         
             min  mean max    min mean max
animal                                    
alligator     13  13.5  14      5  5.5   6
cat            8  10.4  14      1  5.2   9
hamster        7   9.0  10      3  6.0   9
snake         11  11.0  11      6  7.0   8

### 2.2 Diccionario de funciones (funciones diferentes por columna)

In [38]:
# 2.2 Diccionario: funciones distintas para cada columna
# Incluso se pueden usar funciones lambda
df.groupby("animal").agg({
    'weight': ['mean', 'max'], 
    'length': 'std', 
    'age': lambda x: np.percentile(x, 50)
})

weight        length      age
            mean max       std <lambda>
animal                                 
alligator   13.5  14  0.707107      8.0
cat         10.4  14  2.949576     10.0
hamster      9.0  10  2.449490      2.0
snake       11.0  11  1.414214     10.5

### 2.3 Named aggregation (nombres personalizados para las columnas resultado)

Esta es la sintaxis más limpia. Se usan **tuplas** `(columna, funcion)` con **nombres de keyword arguments**:

In [39]:
# 2.3 Named aggregation: columnas con nombres personalizados
df.groupby("animal").agg(
    peso_promedio = ("weight", 'mean'), 
    peso_maximo = ("weight", 'max'),
    edad_mediana = ("age", 'median'),
    num_registros = ("age", 'count')
)

,peso_promedio,peso_maximo,edad_mediana,num_registros
animal,,,,
alligator,13.5,14,8.0,2
cat,10.4,14,10.0,5
hamster,9.0,10,2.0,5
snake,11.0,11,10.5,2


---
## 3. Agrupación por múltiples columnas

Se puede agrupar por **más de una columna** pasando una lista a `.groupby()`. Esto crea subgrupos más granulares.

In [40]:
# Agreguemos una columna categórica al ejemplo de animales para ilustrar
df['tamaño'] = df['weight'].apply(lambda x: 'grande' if x >= 11 else 'pequeño')
df

,animal,age,weight,length,tamaño
0,hamster,1,7,8,pequeño
1,alligator,9,13,6,grande
2,hamster,4,8,9,pequeño
3,cat,13,12,1,grande
4,snake,14,11,8,grande
5,cat,10,8,9,pequeño
6,hamster,2,10,5,pequeño
7,cat,4,14,6,grande
8,cat,14,9,6,pequeño
9,snake,7,11,6,grande


In [41]:
# Agrupar por animal Y tamaño
df.groupby(["animal", "tamaño"]).agg(
    edad_promedio = ("age", "mean"),
    conteo = ("age", "count")
)

edad_promedio  conteo
animal    tamaño                        
alligator grande        8.000000       2
cat       grande        8.500000       2
          pequeño       8.333333       3
hamster   pequeño       4.600000       5
snake     grande       10.500000       2

> **Tip:** Cuando se agrupa por múltiples columnas, el resultado tiene un **MultiIndex**. Se puede usar `.reset_index()` para convertirlo en un DataFrame plano, lo cual facilita filtros y visualizaciones posteriores.

In [42]:
# reset_index() convierte el resultado en un DataFrame plano
df.groupby(["animal", "tamaño"]).agg(
    edad_promedio = ("age", "mean"),
    conteo = ("age", "count")
).reset_index()

,animal,tamaño,edad_promedio,conteo
0,alligator,grande,8.000000,2
1,cat,grande,8.500000,2
2,cat,pequeño,8.333333,3
3,hamster,pequeño,4.600000,5
4,snake,grande,10.500000,2


---
## Tarea --- Accidentes de tránsito en Bogotá (2016)

**Base de datos:** `info_accidentes.csv`  
**Formato:** Grupos de 4 personas  
**Entrega:** Notebook (.ipynb) con código y respuestas

**Calificación sobre 5.0 (10 preguntas x 0.5 pts):**

| Nivel | Puntos | Preguntas |
|---|---|---|
| Nivel 1: Exploración e inspección | 1.5 | 1--3 |
| Nivel 2: Agrupaciones con `groupby` | 2.0 | 4--7 |
| Nivel 3: Análisis avanzado con `.agg()` | 1.5 | 8--10 |
| **Total** | **5.0** | |

---

### Nivel 1: Exploración e inspección (1.5 puntos)

**Pregunta 1 (0.5 pts):** Cargue la base de datos `info_accidentes.csv` e inspeccione su estructura. Reporte:
- ¿Cuántos registros (filas) y cuántas variables (columnas) tiene la base?
- ¿Qué tipos de datos tienen las columnas?
- ¿Hay valores nulos en alguna columna?

In [ ]:
# PREGUNTA 1:
df_accidentes = pd.read_csv("C:/Users/David/Downloads/Clase 10/data/info_accidentes.csv")
df_accidentes.shape
df_accidentes


(34931, 25)

In [63]:
df_accidentes.info()

<class 'pandas.DataFrame'>
RangeIndex: 34931 entries, 0 to 34930
Data columns (total 25 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Fecha             34931 non-null  str    
 1   GravedadNombre    34931 non-null  str    
 2   ClaseNombre       34931 non-null  str    
 3   ChoqueNombre      34931 non-null  str    
 4   ObjetoFijoCodigo  34931 non-null  str    
 5   ObjetoFijoNombre  34931 non-null  str    
 6   OtraClase         34931 non-null  str    
 7   NombreOtraClase   34931 non-null  str    
 8   Latitud           34931 non-null  float64
 9   Longitud          34931 non-null  float64
 10  Direccion         34931 non-null  str    
 11  TipoVia1          34931 non-null  str    
 12  NumeroVia1        34931 non-null  str    
 13  LetraVia1         34931 non-null  str    
 14  CardinalVia1      34931 non-null  str    
 15  TipoVia2          34931 non-null  str    
 16  NumeroVia2        34625 non-null  float64
 17  Letr

**Pregunta 2 (0.5 pts):** ¿Cuántos muertos hay registrados **en total** en la base de datos? ¿Y cuántos heridos en total? ¿Cuántos accidentes hubo en la localidad de **Kennedy**? ¿Y en **Chapinero**?

In [ ]:
# PREGUNTA 2:
df_accidentes.TotalMuertos.sum()


np.int64(575)

In [ ]:
df_accidentes.TotalHeridos.sum()


In [ ]:
localidades = df_accidentes.groupby("Localidad")
localidades.size()

# Kennedy = 4009
# Chapi = 2418

Localidad
ANTONIO NARIÑO         666
BARRIOS UNIDOS        1885
BOSA                  1524
CANDELARIA             174
CHAPINERO             2418
CIUDAD BOLIVAR        1389
ENGATIVA              3487
FONTIBON              2793
KENNEDY               4009
LOS MARTIRES          1176
PUENTE ARANDA         2409
RAFAEL URIBE URIBE     933
SAN CRISTOBAL          863
SANTA FE               967
SUBA                  3336
TEUSAQUILLO           1778
TUNJUELITO             932
USAQUEN               3538
USME                   654
dtype: int64

**Pregunta 3 (0.5 pts):** ¿Cuál es la distribución de los accidentes según su gravedad (`GravedadNombre`)? Es decir, ¿cuántos accidentes hay de cada tipo de gravedad? Muestre también la proporción (%) de cada categoría.

In [75]:
# PREGUNTA 3:
df_accidentes["GravedadNombre"].value_counts(normalize=True)

GravedadNombre
Solo Daños     0.681859
Con Heridos    0.302253
Con Muertos    0.015888
Name: proportion, dtype: float64

In [76]:
df_accidentes["GravedadNombre"].value_counts()

GravedadNombre
Solo Daños     23818
Con Heridos    10558
Con Muertos      555
Name: count, dtype: int64

---
### Nivel 2: Agrupaciones con `groupby` (2.0 puntos)

**Pregunta 4 (0.5 pts):** Usando `groupby`, calcule el **número total de accidentes, muertos y heridos** por cada **localidad**. ¿Cuál es la localidad con más muertos? ¿Y la localidad con más heridos? ¿Son la misma?

In [ ]:
# PREGUNTA 4:
df_accidentes.groupby(["Localidad"]).agg(
    Accidentes = ("Localidad", "count"),
    Muertos = ("TotalMuertos", "sum"),
    Heridos = ("TotalHeridos", "sum")
)

# Localidad con más muertos es Kennedy
# Localidad con más heridos es Kennedy

,Accidentes,Muertos,Heridos
Localidad,,,
ANTONIO NARIÑO,666,19,361
BARRIOS UNIDOS,1885,23,627
BOSA,1524,33,855
CANDELARIA,174,2,65
CHAPINERO,2418,15,628
CIUDAD BOLIVAR,1389,39,883
ENGATIVA,3487,63,1242
FONTIBON,2793,39,832
KENNEDY,4009,78,1939


**Pregunta 5 (0.5 pts):** Agrupe los datos por **clase de accidente** (`ClaseNombre`) y calcule:
- El número de accidentes de cada clase
- El promedio de heridos por accidente de cada clase
- El total de muertos por cada clase

¿Qué clase de accidente tiene el mayor promedio de heridos por accidente? ¿Y cuál genera más muertos en total?

In [87]:
# PREGUNTA 5:
df_accidentes.groupby(["ClaseNombre"]).agg(
    NumeroAccidentes = ("ClaseNombre", "count"),
    PromedioHeridos = ("TotalHeridos", "mean"),
    TotalMuertos = ("TotalMuertos", "sum")
)


,NumeroAccidentes,PromedioHeridos,TotalMuertos
ClaseNombre,,,
Atropello,3669,1.162442,265
Autolesion,4,1.000000,0
Caida de ocupante,869,1.100115,9
Choque,29949,0.291863,260
Incendio,1,0.000000,0
Otro,92,0.978261,0
Volcamiento,347,1.097983,41


**Pregunta 6 (0.5 pts):** Agrupe los datos por **condición climática** (`TipoTiempo`) y calcule el total de accidentes y el total de muertos para cada condición. ¿En qué condición climática ocurren más accidentes? ¿Se puede concluir que esa condición climática **causa** más accidentes? Justifique brevemente.

In [ ]:
# PREGUNTA 6:
df_accidentes.groupby(["TipoTiempo"]).agg(
    Accidentes = ("TipoTiempo", "count"),
    Muertos = ("TotalMuertos", "sum")
)
# En la condición normal ocurren más accidentes
# No se puede atribuir causalidad ya que la mayor parte del tiempo el tiempo es de tipo normal

,Accidentes,Muertos
TipoTiempo,,
,10,0
Lluvia,1214,13
Lluvia/Lluvia,2,0
Lluvia/Normal,7,0
Niebla,35,0
Normal,33607,562
Normal/Lluvia,2,0
Normal/Normal,10,0
Viento,43,0


**Pregunta 7 (0.5 pts):** Agrupe los datos por **tipo de diseño vial** (la columna que hace referencia al diseño de la vía) y calcule el número de accidentes, el total de muertos y el promedio de heridos por accidente. ¿Qué tipo de diseño vial concentra la mayor cantidad de accidentes? ¿Y cuál tiene el mayor promedio de heridos por accidente?

In [ ]:
# PREGUNTA 7:
df_accidentes.groupby(["TipoDiseño"]).agg(
    Accidentes = ("TipoDiseño", "count"),
    Muertos = ("TotalMuertos", "sum"),
    PromedioHeridos = ("TotalHeridos", "mean")
)
# Tramo de vía es el tipo de diseño con más accidentes
# Cicloruta es el tipo de diseño con mayor promedio de heridos


,Accidentes,Muertos,PromedioHeridos
TipoDiseño,,,
Cicloruta,22,0,1.136364
Glorieta,396,2,0.196970
Interseccion,7458,91,0.503352
Lote o predio,313,4,0.150160
Paso a nivel,79,0,0.202532
Paso elevado,267,2,0.303371
Paso inferior,211,1,0.308057
Ponton,2,0,0.000000
Puente,96,0,0.416667


---
### Nivel 3: Análisis avanzado con `.agg()` (1.5 puntos)

**Pregunta 8 (0.5 pts):** Realice un `groupby` cruzando **dos variables**: `Localidad` y `GravedadNombre`. Para cada combinación, calcule el número de accidentes. Luego responda:

- ¿Cuál es la localidad con mayor número de accidentes **con muertos**?
- ¿Cuál es la localidad con mayor número de accidentes que resultaron **solo en daños**?
- ¿Son la misma localidad? ¿Qué podría explicar la diferencia?

In [105]:
# PREGUNTA 8:
df_con_muertos = df_accidentes[df_accidentes["GravedadNombre"] == "Con Muertos"]
df_con_muertos.groupby(["Localidad", "GravedadNombre"]).agg(
    NumeroAccidentes = ("Localidad", "count"),
).reset_index().sort_values(by="NumeroAccidentes" , ascending=False)
# La localidad con mayor número de accidentes con muertos es Kennedy
# La localidad con mayor número de accidentes con solo daños es Kennedy

,Localidad,GravedadNombre,NumeroAccidentes
8,KENNEDY,Con Muertos,77
6,ENGATIVA,Con Muertos,60
14,SUBA,Con Muertos,41
10,PUENTE ARANDA,Con Muertos,40
7,FONTIBON,Con Muertos,38
5,CIUDAD BOLIVAR,Con Muertos,37
17,USAQUEN,Con Muertos,35
2,BOSA,Con Muertos,32
16,TUNJUELITO,Con Muertos,25
11,RAFAEL URIBE URIBE,Con Muertos,23


**Pregunta 9 (0.5 pts):** Usando `.agg()` con **named aggregation**, construya una tabla resumen agrupada por `Localidad` que contenga las siguientes columnas:

| Columna resultado | Operación |
|---|---|
| `total_accidentes` | Conteo de accidentes |
| `total_muertos` | Suma de `TotalMuertos` |
| `total_heridos` | Suma de `TotalHeridos` |
| `promedio_heridos` | Promedio de `TotalHeridos` |
| `max_heridos_un_accidente` | Máximo de `TotalHeridos` |

Ordene la tabla por `total_muertos` de mayor a menor. Interprete brevemente: ¿la localidad con más accidentes es la misma que la que tiene más muertos? ¿Qué localidad tiene el mayor número de heridos en **un solo accidente**?

In [51]:
# PREGUNTA 9:

**Pregunta 10 (0.5 pts):** Agrupe los datos simultáneamente por `ClaseNombre` y `Localidad`. Usando `.agg()`, calcule el total de muertos y el total de heridos para cada combinación. ¿Cuáles son las 5 combinaciones de clase de accidente y localidad con mayor número de muertos?

In [52]:
# PREGUNTA 10: